# OmniVoice Fixed Notebook
### ✔ No freeze | ✔ Stable Tunnel | ✔ Server logging

In [ ]:
%cd /content/
!rm -rf omnivoice-colab
!git clone https://github.com/hahyt6644-gif/omnivoice-colab.git
%cd omnivoice-colab

!rm -rf OmniVoice
!git clone https://github.com/k2-fsa/OmniVoice.git

# Install cloudflared
!curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/bin/cloudflared
!chmod +x /usr/bin/cloudflared

!apt-get update -y && apt-get install -y ffmpeg
!pip install -q fastapi uvicorn[standard] python-multipart httpx requests aiofiles nest-asyncio pydub soundfile librosa
!pip install -q -r colab.txt

from IPython.display import clear_output
clear_output()
print('✅ INSTALLATION COMPLETE')

In [ ]:
import subprocess, time, re, requests, os, threading

# Cleanup any existing instances
!fuser -k 7860/tcp

print('🚀 Starting OmniVoice server...')
# Output redirected to a log file so we can see if it crashes
server = subprocess.Popen(['python', 'app.py'], stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)

# Wait for server
server_ready = False
for i in range(30):
    try:
        # Use 127.0.0.1 instead of localhost for better Colab compatibility
        r = requests.get('http://127.0.0.1:7860', timeout=2)
        print('✅ Server ready')
        server_ready = True
        break
    except:
        time.sleep(2)

if not server_ready:
    print("❌ Server failed to start. Last 20 lines of server log:")
    os.system("tail -n 20 server.log")
else:
    print('🌍 Starting tunnel...')
    tunnel = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860'], 
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )

    url = None
    start = time.time()
    while time.time() - start < 30:
        line = tunnel.stdout.readline()
        if not line:
            continue
        
        if 'trycloudflare.com' in line:
            match = re.search(r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com', line)
            if match:
                url = match.group(0)
                break

    if url:
        print('\n========================')
        print(f'✅ PUBLIC URL: {url}')
        print(f'UI: {url}/ui')
        print('========================')
        
        # PREVENT FREEZE: Start a background thread to continually consume tunnel output
        def consume_tunnel_output(tunnel_proc):
            for _ in tunnel_proc.stdout:
                pass
        threading.Thread(target=consume_tunnel_output, args=(tunnel,), daemon=True).start()
        
    else:
        print('❌ Tunnel failed. Run this cell again.')

In [ ]:
import nest_asyncio
nest_asyncio.apply()
print('Stable runtime active.')